# Multivariatize a univariate generator

`Multivariatizer` turns any univariate generator into a set of *cross-dependent* channels — the realistic case where series move together (a store's product lines, correlated sensors, coupled macro indicators) rather than in isolation. It draws independent series from the wrapped generator, couples them, and preserves the wrapped generator's per-series diversity.

::: {.callout-note}
## Two couplings

- **`mixing`** (contemporaneous): each output channel becomes a linear blend of the base series through a well-conditioned mixing matrix — instantaneous correlation.
- **`leadlag`** (temporal): a channel becomes a lagged, scaled, noise-perturbed copy of another — one series leads, the other follows.

Both are sampled per call and can compose. For processes that are multivariate *by construction*, see the [multivariate generators](../generators/multivariate/var) instead.
:::

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from synforecast import Multivariatizer
from synforecast.generators import TSIGenerator

In [ ]:
base = TSIGenerator(
    min_length=256,
    max_length=256,
    freq="h",
    engine="polars",
    seed=7,
)
coupled = Multivariatizer(base=base, seed=42)
df = coupled.generate(n_series=4)
df.head()

## Inspect the recipe

Every panel records exactly how it was built — the mixing matrix and each lead-lag edge — so a generated dataset is reproducible from its seed and auditable after the fact.

In [ ]:
coupled.last_recipe

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for series_id in df["unique_id"].unique().sort().to_list():
    series = df.filter(df["unique_id"] == series_id).sort("ds")
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=series_id)
ax.set(title="Coupled synthetic series", xlabel="Time", ylabel="y")
ax.legend()
plt.show()

In [ ]:
wide = df.pivot(on="unique_id", index="ds", values="y").sort("ds")
np.corrcoef(wide.drop("ds").to_numpy().T)

The off-diagonal terms are non-zero and asymmetric: the couplings induced genuine cross-series dependence, including the negative lead-lag link between channels 2 and 3 recorded in the recipe above. Because the whole panel derives from one seed, this structure regenerates exactly.